情况3：设置为自定义错误处理函数

In [1]:
from typing import Union

from langchain.agents.structured_output import ToolStrategy,MultipleStructuredOutputsError,StructuredOutputValidationError
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from rich import print as rprint
# 1.初始化模型
model = ChatOpenAI(
    # model="carstenuhlig/omnicoder-9b:latest",
    model="483025889/qwen3.5:9b",
    # model="mistral-nemo:latest",
    # model="qwen3-vl:latest",
    api_key="sk12345",
    base_url="http://localhost:11434/v1",
)

## 定义错误处理函数，太复杂，会引起agent死循环
# def custom_error_handler(error: Exception) ->str:
#     """自定义错误处理器"""
#     err_str = str(error)
#     print(f"捕获到错误类型{type(error).__name__}")
#     print(f"错误详情{err_str}")
#     if isinstance(error, MultipleStructuredOutputsError):
#         return "检测到多个响应，请选择最相关的一个进行返回"
#     # elif isinstance(error, StructuredOutputValidationError):
#     #     return "评价数据格式有误，请检查字段是否符合要求."
#
#     else:
#         return f"Error: {err_str}"

## 定义错误处理函数，太复杂，会引起agent死循环
# def custom_error_handler2(error: Exception) ->str:
#     """自定义错误处理器"""
#     if isinstance(error, MultipleStructuredOutputsError):
#         return "检测到多个响应，请选择最相关的一个进行返回"
#     else:
#         return f"Error: {str(error)}"
    
def custom_error_handler3(error: Exception) ->str:
    """自定义错误处理器"""
    return f"Error: {str(error)}"    


class ContactInfo(BaseModel):
    """联系人信息"""
    name: str = Field(description="姓名")
    email: str = Field(description="邮箱")

class EnventDetails(BaseModel):
    """活动详情"""
    event_name: str = Field(description="活动名称")
    date: str = Field(description="活动日期")

agent = create_agent(
    model=model,
    response_format=ToolStrategy(
        Union[ContactInfo, EnventDetails], ## 注意Union里面的东西只能取一个
        tool_message_content="提取完成！",
        handle_errors=custom_error_handler3
    )
)


result = agent.invoke({
    "messages":[{
        "role":"user",
        "content":"请提取以下文本中的内容：姓名:张三，电子邮箱：zhang3@atguigu.com,活动名称：公司年会，活动日期：2026-7-15"
    }]
})

rprint(result)

{
    'messages': [
        HumanMessage(
            content='请提取以下文本中的内容：姓名:张三，电子邮箱：zhang3@atguigu.com,活动名称：公司年会，活动日期：
2026-7-15',
            additional_kwargs={},
            response_metadata={},
            id='96a3d79a-966f-408f-816a-74c0255f040f'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 293,
                    'prompt_tokens': 408,
                    'total_tokens': 701,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': '483025889/qwen3.5:9b',
                'system_fingerprint': 'fp_ollama',
                'id': 'chatcmpl-619',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a11ed2-bb58-7453-bbf3-98251658b53e-0',
            tool_calls=[
                {
                    'name': 'ContactInfo',
                    'args': {'name': '张三', 'email': 'zhang3@atguigu.com'},
                    'id': 'call_kqkce35k',
                    'type': 'tool_call'
                },
                {
                    'name': 'EnventDetails',
                    'args': {'event_name': '公司年会', 'date': '2026-7-15'},
                    'id': 'call_0irbsqiv',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 408,
                'output_tokens': 293,
                'total_tokens': 701,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content='Error: Model incorrectly returned multiple structured responses (ContactInfo, EnventDetails) 
when only one is expected.',
            name='ContactInfo',
            id='3f87b4e5-a5d8-40c2-a7a2-f7b483d08dcc',
            tool_call_id='call_kqkce35k'
        ),
        ToolMessage(
            content='Error: Model incorrectly returned multiple structured responses (ContactInfo, EnventDetails) 
when only one is expected.',
            name='EnventDetails',
            id='846c2678-169d-4eed-b2c5-28e603e59be6',
            tool_call_id='call_0irbsqiv'
        ),
        AIMessage(
            content='根据您提供的文本内容，我已经提取到以下信息：\n\n**联系人信息：**\n- 姓名：张三\n- 
电子邮箱：zhang3@atguigu.com\n\n**活动详情：**\n- 活动名称：公司年会\n- 活动日期：2026-7-15',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 130,
                    'prompt_tokens': 566,
                    'total_tokens': 696,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 404}
                },
                'model_provider': 'openai',
                'model_name': '483025889/qwen3.5:9b',
                'system_fingerprint': 'fp_ollama',
                'id': 'chatcmpl-166',
                'finish_reason': 'stop',
                'logprobs': None
            },
            id='lc_run--01a11ed4-3eaf-76f1-af3c-6ab9e1e5b384-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 566,
                'output_tokens': 130,
                'total_tokens': 696,
                'input_token_details': {'cache_read': 404},
                'output_token_details': {}
            }
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 1,
                    'prompt_tokens': 632,
                    'total_tok